# Chemical Space Split Visualization

This notebook visualizes Morgan fingerprint chemical space for random and scaffold splits. Points are colored by train, validation, and test assignment to show how scaffold splitting creates a stricter chemical-family generalization setting.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.visualize import build_chemical_space_dataframe, plot_chemical_space

DATA_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REDUCTION_METHOD = "t-SNE"
REDUCTION_SEED = 42
FINGERPRINT_BITS = 512
TSNE_PERPLEXITY = 30

datasets = {
    "esol": pd.read_csv(DATA_DIR / "esol.csv"),
    "freesolv": pd.read_csv(DATA_DIR / "freesolv.csv"),
}

{dataset: df.shape for dataset, df in datasets.items()}

{'esol': (1128, 2), 'freesolv': (642, 2)}

## Random vs Scaffold Split Assignments

Each dataset is embedded once per split type using 512-bit Morgan fingerprints and t-SNE with fixed seed 42. The saved files use `chemical_space_<dataset>_<split>.png`.

In [2]:
saved_figures = []

for dataset, df in datasets.items():
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, split_type in zip(axes, ["random", "scaffold"]):
        chemical_space = build_chemical_space_dataframe(
            df,
            split_type=split_type,
            seed=REDUCTION_SEED,
            n_bits=FINGERPRINT_BITS,
            perplexity=TSNE_PERPLEXITY,
        )
        plot_chemical_space(
            chemical_space,
            dataset=dataset,
            split_type=split_type,
            ax=ax,
        )

        single_fig, single_ax = plt.subplots(figsize=(6.5, 5.5))
        plot_chemical_space(
            chemical_space,
            dataset=dataset,
            split_type=split_type,
            ax=single_ax,
        )
        output_path = FIGURES_DIR / f"chemical_space_{dataset}_{split_type}.png"
        single_fig.tight_layout()
        single_fig.savefig(output_path, dpi=160)
        plt.close(single_fig)
        saved_figures.append(str(output_path.relative_to(PROJECT_ROOT)))

    fig.suptitle(
        f"{dataset.upper()} chemical space split comparison ({REDUCTION_METHOD}, seed={REDUCTION_SEED})",
        y=1.02,
    )
    fig.tight_layout()
    plt.show()

saved_figures

/var/folders/76/n9t9gpfd0fv2h7d9d4ypy0380000gn/T/ipykernel_61953/1555273311.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/76/n9t9gpfd0fv2h7d9d4ypy0380000gn/T/ipykernel_61953/1555273311.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


['results/figures/chemical_space_esol_random.png',
 'results/figures/chemical_space_esol_scaffold.png',
 'results/figures/chemical_space_freesolv_random.png',
 'results/figures/chemical_space_freesolv_scaffold.png']